# Premotor neuronal activity

Run with `movement_bouts/` as the kernel's working directory.

`movement_bouts` is the reference table for all behavioral metadata. The neural
output has one `spike_counts` array per bout, indexed by the **same original bout
index**. Each array has shape `(neurons, 5 ms time bins)` and excludes movement onset.
Neuron IDs are stored once per session, indexed by `session_info.index` (`idx`).

See [the shared analysis format](ANALYSIS_DATA_FORMAT.md) for the data/labels
interface used by PCA, future UMAP analyses, and evaluation helpers in `evaluation.py`.


# 1. Load the reference tables and define parameters


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

from importlib import reload
import utils
reload(utils)

from utils import (
    extract_all_premotor_bouts, prepare_pca_input,
    fit_pca_by_condition, plot_pca_session,
)

data_dir = Path("../../data").resolve()
results_dir = data_dir / "derived" / "movement_bouts"
premotor_dir = data_dir / "derived" / "premotor_activity"

movement_bouts = pd.read_pickle(results_dir / "movement_bouts.pkl")
session_info = pd.read_pickle(results_dir / "session_info.pkl")
parameters = json.loads((results_dir / "parameters.json").read_text(encoding="utf-8"))

print(f"Loaded {len(movement_bouts)} movement bouts across {len(session_info)} session entries.")


In [ ]:
DT = parameters["DT"]
EXTRACTION_WINDOW_S = (-1.2, 0.0)  # Raw 5 ms counts to retain around each onset.
ANALYSIS_WINDOW_S = (-0.5, 0.0)  # Only this interval enters PCA/evaluation.
BASELINE_WINDOW_S = (-1.0, -0.5)  # Mean activity per bout and neuron.
ACTIVITY_MODE = "baseline_subtracted"  # "raw" or "baseline_subtracted".
N_BINS = 5  # Within ANALYSIS_WINDOW_S: 5 = 100 ms, 2 = 250 ms, 1 = 500 ms.

if ACTIVITY_MODE not in ("raw", "baseline_subtracted"):
    raise ValueError("Choose raw or baseline_subtracted activity.")
if EXTRACTION_WINDOW_S[0] < -parameters["MIN_IMMOBILITY_S"]:
    raise ValueError("Extraction extends before the guaranteed immobility interval.")

extraction_dir = premotor_dir / f"extracted_{EXTRACTION_WINDOW_S[0]:g}_{EXTRACTION_WINDOW_S[1]:g}"
run_name = (
    f"analysis_{ANALYSIS_WINDOW_S[0]:g}_{ANALYSIS_WINDOW_S[1]:g}_{N_BINS}bins_{ACTIVITY_MODE}"
)
if ACTIVITY_MODE == "baseline_subtracted":
    run_name += f"_{BASELINE_WINDOW_S[0]:g}_{BASELINE_WINDOW_S[1]:g}"
run_dir = premotor_dir / "runs" / run_name
pca_dir = run_dir / "pca"
evaluation_dir = run_dir / "evaluation"
pca_path = pca_dir / f"pca_{N_BINS}bins.pkl"
tracking_projection_path = pca_dir / f"pca_tracking_projection_{N_BINS}bins.pkl"
print(f"Activity mode: {ACTIVITY_MODE} | results: {run_dir}")


# 2. Extract and save the original-resolution counts

The helper loads each source collection once, then extracts session by session.
It skips bouts without a complete window and records their indices and reasons.
Behavioral metadata stays in `movement_bouts`; its indices are never reset.

To extract only one session, pass `session_info.loc[[idx]]` instead of `session_info`.


In [ ]:
neuronal_bouts, neuron_ids_by_session, excluded_bouts = extract_all_premotor_bouts(
    data_dir, movement_bouts, session_info,
    dt=DT, extraction_window_s=EXTRACTION_WINDOW_S,
)

print(f"Extracted {len(neuronal_bouts)} bouts; skipped {len(excluded_bouts)}.")
if len(excluded_bouts):
    display(excluded_bouts.head())


In [ ]:
import hashlib
from datetime import datetime, timezone

extraction_dir.mkdir(parents=True, exist_ok=True)
neuronal_bouts.to_pickle(extraction_dir / "neuronal_bouts.pkl")
neuron_ids_by_session.to_pickle(extraction_dir / "neuron_ids_by_session.pkl")
excluded_bouts.to_pickle(extraction_dir / "excluded_bouts.pkl")

# Fingerprints link bout indices to the exact reference tables used here.
extraction_parameters = {
    "DT": DT,
    "window_s": list(EXTRACTION_WINDOW_S),
    "n_time_bins": round((EXTRACTION_WINDOW_S[1] - EXTRACTION_WINDOW_S[0]) / DT),
    "onset_included": False,
    "units": "spike counts per original bin",
    "session_indices": [int(idx) for idx in neuron_ids_by_session.index],
    "source_directory": str(results_dir),
    "source_sha256": {
        name: hashlib.sha256((results_dir / name).read_bytes()).hexdigest()
        for name in ("movement_bouts.pkl", "session_info.pkl")
    },
    "bout_detection_parameters": parameters,
    "saved_at_utc": datetime.now(timezone.utc).isoformat(),
}
(extraction_dir / "parameters.json").write_text(
    json.dumps(extraction_parameters, indent=2) + "\n", encoding="utf-8"
)
print(f"Saved premotor tables to {extraction_dir}")


# 3. Prepare one session for PCA

Load the saved wider counts and their timing metadata. Section 2 is needed only
when the extraction interval or reference bout table changes. Every row keeps its
original bout ID; the counts remain unnormalized.

`prepare_pca_input()` selects ANALYSIS_WINDOW_S, averages into N_BINS windows,
and optionally subtracts the mean BASELINE_WINDOW_S rate for each bout/neuron.
It returns the same aligned pca_input/pca_labels tables as before. Negative
corrected rates mean activity below baseline; values remain in spikes/s.

Changing ACTIVITY_MODE, the analysis/baseline interval, or N_BINS requires rerunning
the constants cell and preparation/PCA/evaluation cells. Within the saved extraction,
these changes do not require loading the original recordings again. Each analysis
choice has its own result directory. Raw is the default for comparison with earlier
results; choose baseline_subtracted to use the correction throughout the notebook.


In [ ]:
extraction_parameters = json.loads((extraction_dir / "parameters.json").read_text(encoding="utf-8"))
neuronal_bouts = pd.read_pickle(extraction_dir / "neuronal_bouts.pkl")
neuron_ids_by_session = pd.read_pickle(extraction_dir / "neuron_ids_by_session.pkl")

idx = 0
pca_input, pca_labels = prepare_pca_input(
    neuronal_bouts, movement_bouts, neuron_ids_by_session,
    idx=idx, dt=extraction_parameters["DT"], n_bins=N_BINS,
    extraction_window_s=extraction_parameters["window_s"],
    analysis_window_s=ANALYSIS_WINDOW_S, baseline_window_s=BASELINE_WINDOW_S,
    activity_mode=ACTIVITY_MODE,
)

print(f"idx={idx}: {ACTIVITY_MODE} input {pca_input.shape} (bout-window observations, neurons)")
display(pca_input.head(N_BINS))
display(pca_labels.head(N_BINS))


### Inspect the baseline correction for the same bout

The right heatmap is centered on zero. The displayed baseline is subtracted from every analysis bin of its neuron.


In [ ]:
import matplotlib.pyplot as plt

raw_example, _ = prepare_pca_input(
    neuronal_bouts, movement_bouts, neuron_ids_by_session,
    idx=idx, dt=extraction_parameters["DT"], n_bins=N_BINS,
    extraction_window_s=extraction_parameters["window_s"], analysis_window_s=ANALYSIS_WINDOW_S,
    activity_mode="raw",
)
corrected_example, _ = prepare_pca_input(
    neuronal_bouts, movement_bouts, neuron_ids_by_session,
    idx=idx, dt=extraction_parameters["DT"], n_bins=N_BINS,
    extraction_window_s=extraction_parameters["window_s"], analysis_window_s=ANALYSIS_WINDOW_S,
    baseline_window_s=BASELINE_WINDOW_S, activity_mode="baseline_subtracted",
)
example_bout_id = raw_example.index.get_level_values("bout_id").unique()[0]
raw_bout = raw_example.xs(example_bout_id, level="bout_id")
corrected_bout = corrected_example.xs(example_bout_id, level="bout_id")
baseline_example = (raw_bout - corrected_bout).iloc[0].rename("baseline_rate_spikes_s")
display(baseline_example.to_frame())
example_edges_ms = np.append(raw_bout.index.to_numpy(), ANALYSIS_WINDOW_S[1]) * 1000
example_limit = max(float(np.abs(corrected_bout.to_numpy()).max()), 1e-12)
fig_baseline_example, axes_baseline_example = plt.subplots(
    1, 2, figsize=(14, max(4, len(raw_bout.columns) * 0.3)), sharex=True, sharey=True,
)
for ax, values, name, colors in zip(
    axes_baseline_example, (raw_bout, corrected_bout),
    ("Raw firing rate", "Baseline-subtracted firing rate"),
    (dict(cmap="viridis", vmin=0), dict(cmap="RdBu_r", vmin=-example_limit, vmax=example_limit)),
):
    heatmap = ax.pcolormesh(
        example_edges_ms, np.arange(len(values.columns) + 1) - 0.5,
        values.to_numpy().T, shading="flat", **colors,
    )
    ax.set(title=name, xlabel="Time relative to movement onset (ms)",
           yticks=np.arange(len(values.columns)), yticklabels=values.columns)
    fig_baseline_example.colorbar(heatmap, ax=ax, label="spikes/s")
axes_baseline_example[0].set_ylabel("Neuron")
axes_baseline_example[0].invert_yaxis()
fig_baseline_example.suptitle(f"idx={idx} | bout {example_bout_id} | baseline {BASELINE_WINDOW_S} s")
fig_baseline_example.tight_layout()
plt.show()


# 4. Inspect one bout: all neurons and averaging windows


In [ ]:
import matplotlib.pyplot as plt

# Choose one retained bout by its position within the prepared session.
bout_number = 0
bout_id = pca_input.index.get_level_values("bout_id").unique()[bout_number]
bout_rates = pca_input.xs(bout_id, level="bout_id")
bout_labels = pca_labels.xs(bout_id, level="bout_id")
condition = bout_labels["condition"].iloc[0]
condition_name = {0: "Tracking", 1: "Playback"}.get(condition, str(condition))
if ACTIVITY_MODE == "baseline_subtracted":
    color_limit = max(float(np.abs(bout_rates.to_numpy()).max()), 1e-12)
    heatmap_colors = dict(cmap="RdBu_r", vmin=-color_limit, vmax=color_limit)
else:
    heatmap_colors = dict(cmap={0: "Reds", 1: "Greys"}.get(condition, "viridis"), vmin=0)
edges_ms = np.append(bout_rates.index.to_numpy(), bout_labels["bin_end_s"].iloc[-1]) * 1000
window_ms = edges_ms[1] - edges_ms[0]

fig, ax = plt.subplots(figsize=(11, max(4, len(bout_rates.columns) * 0.3)))
heatmap = ax.pcolormesh(
    edges_ms, np.arange(len(bout_rates.columns) + 1) - 0.5,
    bout_rates.to_numpy().T, shading="flat", **heatmap_colors,
)
ax.set(
    yticks=np.arange(len(bout_rates.columns)), yticklabels=bout_rates.columns,
    xlabel="Time relative to movement onset (ms)", ylabel="Neuron",
    title=f"idx={idx} | bout ID {bout_id} | {condition_name} | {window_ms:g} ms windows | {ACTIVITY_MODE}",
    xlim=(edges_ms[0], edges_ms[-1]),
)
ax.invert_yaxis()
fig.colorbar(heatmap, ax=ax, label=f"{ACTIVITY_MODE} firing rate (spikes/s)")
fig.tight_layout()
plt.show()


# 5. Fit separate PCAs for tracking and playback

`fit_pca_by_condition()` fits the two conditions independently, retaining all
components without variance standardization or whitening. `pca_fits[0]` is
tracking and `pca_fits[1]` is playback. Each entry contains a fitted `model` and
`results` (PCA coordinates joined to the original bout/window labels).


In [ ]:
pca_fits = fit_pca_by_condition(pca_input, pca_labels)
for condition, fit in pca_fits.items():
    name = {0: "Tracking", 1: "Playback"}[condition]
    print(f"{name}, idx={idx}: {len(fit['results'])} observations")
    display(pd.DataFrame(
        {"explained_variance_ratio": fit["model"].explained_variance_ratio_},
        index=[f"PC{i + 1}" for i in range(fit["model"].n_components_)],
    ))


# 6. Plot each PCA in 3D by direction and starting position

Crosses mark leftward movement (-1), circles rightward movement (+1).
Squares mark zero/unknown initial direction, if present. Color represents the
bout starting position, using the same color scale for both separate PCAs.
Every point is a bout/window observation; direction and position are bout-level labels.

The three axes are PC1, PC2, and PC3 from each condition-specific fit.


In [ ]:
import matplotlib.pyplot as plt

fig_pca, axes_pca = plot_pca_session(pca_fits, title=f"idx={idx}: premotor activity | {ACTIVITY_MODE}")
plt.show()


# 7. Compute and save every session's PCAs

Prepare each session using the chosen analysis interval, activity mode and baseline,
then fit tracking/playback separately. pca_cache["sessions"][idx][condition] keeps
the existing model/results structure. No original recording files are loaded.

Save under pca_dir in the selected run directory. Settings retain the actual saved
extraction metadata, analysis/baseline intervals, activity mode, and bin count.
Rerun for each preprocessing/binning choice; both variants can coexist on disk.
To browse a saved run, execute sections 1 and 8 with that run's constants.


In [ ]:
from datetime import datetime, timezone
import sklearn

extraction_parameters = json.loads((extraction_dir / "parameters.json").read_text(encoding="utf-8"))
neuronal_bouts = pd.read_pickle(extraction_dir / "neuronal_bouts.pkl")
neuron_ids_by_session = pd.read_pickle(extraction_dir / "neuron_ids_by_session.pkl")
pca_sessions = {}
skipped_sessions = {}

for session_idx in session_info.index:
    bout_ids = movement_bouts.index[movement_bouts["idx"] == session_idx]
    if not bout_ids.isin(neuronal_bouts.index).any():
        skipped_sessions[session_idx] = "No extracted bouts"
        continue
    session_input, session_labels = prepare_pca_input(
        neuronal_bouts, movement_bouts, neuron_ids_by_session,
        idx=session_idx, dt=extraction_parameters["DT"], n_bins=N_BINS,
        extraction_window_s=extraction_parameters["window_s"],
        analysis_window_s=ANALYSIS_WINDOW_S, baseline_window_s=BASELINE_WINDOW_S,
        activity_mode=ACTIVITY_MODE,
    )
    fits = fit_pca_by_condition(session_input, session_labels)
    if fits:
        pca_sessions[session_idx] = fits
    else:
        skipped_sessions[session_idx] = "Fewer than two observations per condition"

pca_cache = {
    "settings": {
        "DT": extraction_parameters["DT"], "N_BINS": N_BINS,
        "activity_mode": ACTIVITY_MODE,
        "analysis_window_s": list(ANALYSIS_WINDOW_S),
        "baseline_window_s": list(BASELINE_WINDOW_S) if ACTIVITY_MODE == "baseline_subtracted" else None,
        "extraction_directory": str(extraction_dir),
        "standardized": False, "whiten": False,
        "sklearn_version": sklearn.__version__,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "extraction_parameters": extraction_parameters,
    },
    "session_info": session_info.loc[list(pca_sessions)].copy(),
    "sessions": pca_sessions,
    "skipped_sessions": skipped_sessions,
    "position_limits": (
        float(movement_bouts["initial_position"].min()),
        float(movement_bouts["initial_position"].max()),
    ),
}
pca_dir.mkdir(parents=True, exist_ok=True)
pd.to_pickle(pca_cache, pca_path)
print(f"Saved {sum(len(fits) for fits in pca_sessions.values())} PCAs across {len(pca_sessions)} sessions to {pca_path}")
if skipped_sessions:
    print("Skipped sessions:", skipped_sessions)


# 8. Browse PCA by starting position

This viewer has its own session slider. All points are dots, colored by starting
position with fixed limits across sessions. Tracking and playback cameras are
linked within this viewer. Drag to rotate, scroll to zoom, and hover for labels.

Section 9 shows the same saved PCA coordinates colored by direction, with a
separate slider and independent camera controls. No refitting occurs in either.


In [ ]:
from importlib import reload
import utils

reload(utils)
from utils import make_interactive_pca_browser

# Close the former combined viewer and any previous position viewer on rerun.
for browser_name in ("pca_browser", "position_browser"):
    if browser_name in globals():
        old_browser = globals()[browser_name]
        for child in old_browser.children:
            child.close()
        old_browser.close()

pca_cache = pd.read_pickle(pca_path)
position_browser = make_interactive_pca_browser(pca_cache, color_by="position", width=1400)
display(position_browser)


# 9. Browse direction-colored PCA, filtered by starting position

This viewer has independent session and position controls. Select all bouts,
starting left of the midpoint (`initial_position < 14`), or starting at/right of
it (`>= 14`). The midpoint is half of the reconstructed 0-28 position scale;
adjust `midpoint` below if needed. All windows of a bout share its filter.

Dots remain blue for leftward movement and red for rightward movement. Filtering
only hides points in the existing PCA: axes retain their full-session ranges,
models are not refitted, and the two cameras remain linked within this viewer.


In [ ]:
pca_cache = pd.read_pickle(pca_path)

from importlib import reload
import utils

make_interactive_pca_browser = reload(utils).make_interactive_pca_browser
# Close either former direction viewer when this cell is rerun.
for browser_name in ("direction_browser", "filtered_direction_browser"):
    if browser_name in globals():
        old_browser = globals()[browser_name]
        for child in old_browser.children:
            child.close()
        old_browser.close()

direction_browser = make_interactive_pca_browser(
    pca_cache, color_by="direction", filter_position=True, midpoint=14.0, width=1400,
)
display(direction_browser)


# 10. Inspect one bout's premotor trajectory

This independent viewer has a session slider and a bout slider. The selected
bout's windows are labeled by bin number, colored light to dark with time, and
connected in chronological order. Hover gives each window's start and end time.
Other bouts are faint and colored by starting position, using the same fixed
Viridis scale as the position viewer. The selected trajectory keeps its separate
light-to-dark time colors. A bout belongs to one condition, so its highlighted
trajectory appears only in the corresponding tracking or playback panel.

Both cameras stay linked within this viewer. Axis ranges stay fixed for the
session when switching bouts. These are the saved first three PC coordinates;
no PCA is refitted and no new data are saved.


In [ ]:
pca_cache = pd.read_pickle(pca_path)

from importlib import reload
import utils

make_bout_pca_browser = reload(utils).make_bout_pca_browser
if "bout_browser" in globals():
    for child in bout_browser.children:
        child.close()
    bout_browser.close()

bout_browser = make_bout_pca_browser(pca_cache, width=1400)
display(bout_browser)


# 11. Project playback activity into tracking PCA space

Reuse each session's saved **tracking** PCA for both conditions. Playback rates
are transformed with the tracking neuron's mean and loading vectors; they are
not recentered by the playback mean, and no PCA is fitted again. The existing
separate-PCAs results remain unchanged.

Run the computation cell once to save projections for all sessions. On later
visits, run the viewer cell only. Both panels use matching axis limits and linked
cameras: tracking on the left, playback in tracking axes on the right. Color is
starting position. Percentages on both panels describe **tracking** variance,
not the fraction of playback variance explained by those axes.


In [ ]:
from importlib import reload
import utils

project_onto_tracking_pca = reload(utils).project_onto_tracking_pca
reference_pca_path = pca_path
reference_pca_cache = pd.read_pickle(reference_pca_path)
reference_settings = reference_pca_cache["settings"]
reference_extraction_dir = Path(reference_settings["extraction_directory"])
neuronal_bouts = pd.read_pickle(reference_extraction_dir / "neuronal_bouts.pkl")
neuron_ids_by_session = pd.read_pickle(reference_extraction_dir / "neuron_ids_by_session.pkl")
tracking_projection_sessions = {}

for session_idx, fits in reference_pca_cache["sessions"].items():
    if 0 not in fits:
        continue
    session_input, session_labels = utils.prepare_pca_input(
        neuronal_bouts, movement_bouts, neuron_ids_by_session,
        idx=session_idx, dt=reference_pca_cache["settings"]["DT"],
        n_bins=reference_settings["N_BINS"],
        extraction_window_s=reference_settings["extraction_parameters"]["window_s"],
        analysis_window_s=reference_settings["analysis_window_s"],
        baseline_window_s=reference_settings["baseline_window_s"],
        activity_mode=reference_settings["activity_mode"],
    )
    tracking_projection_sessions[session_idx] = project_onto_tracking_pca(
        session_input, session_labels, fits[0]["model"],
    )

tracking_projection_cache = {
    "projection_basis": "tracking",
    "settings": {**reference_pca_cache["settings"], "source_pca_file": reference_pca_path.name},
    "session_info": reference_pca_cache["session_info"].loc[list(tracking_projection_sessions)],
    "position_limits": reference_pca_cache["position_limits"],
    "sessions": tracking_projection_sessions,
}
tracking_projection_path = reference_pca_path.with_name(f"pca_tracking_projection_{reference_settings['N_BINS']}bins.pkl")
pd.to_pickle(tracking_projection_cache, tracking_projection_path)
print(f"Saved tracking-space projections for {len(tracking_projection_sessions)} sessions to {tracking_projection_path}")


In [ ]:
from importlib import reload
import utils

make_interactive_pca_browser = reload(utils).make_interactive_pca_browser
if "tracking_projection_browser" in globals():
    for child in tracking_projection_browser.children:
        child.close()
    tracking_projection_browser.close()

tracking_projection_cache = pd.read_pickle(
    tracking_projection_path
)
tracking_projection_browser = make_interactive_pca_browser(
    tracking_projection_cache, color_by="position", width=1400,
)
display(tracking_projection_browser)


# 12. Browse PCA by destination position

Color each premotor observation by its bout's saved `final_position` (the position
at the movement endpoint). This is an observed destination, not an inferred target.
All windows of a bout share the same destination label. Unfinished bouts with no
known endpoint are gray. The positional scale matches the starting-position view.

This independent session slider uses the existing separate tracking/playback PCAs,
with linked cameras. Only the coloring changes; no extraction or fitting is rerun.


In [ ]:
pca_cache = pd.read_pickle(pca_path)

from importlib import reload
import utils

make_interactive_pca_browser = reload(utils).make_interactive_pca_browser
if "destination_browser" in globals():
    for child in destination_browser.children:
        child.close()
    destination_browser.close()

destination_browser = make_interactive_pca_browser(
    pca_cache, color_by="destination", width=1400,
)
display(destination_browser)


# 13. How closely does destination follow starting position?

Each point is one completed bout from `movement_bouts`, not one premotor window.
Tracking and playback plots compare starting and final positions; the dashed
identity line marks identical endpoints. Titles show Pearson correlation and
median absolute net displacement, in the saved position units.

The third panel is the cumulative distribution of `abs(final_position -
initial_position)`: its height at any distance is the fraction of bouts whose
endpoint is that close to the start. This avoids choosing an arbitrary threshold
for "close". The slider selects a session, matching our session-specific PCAs.
Unknown endpoints are excluded and counted.

Small net displacement can mean a short movement or a return to the starting
position after a longer path. It is not the same as `total_movement`. These are
descriptive associations; they do not isolate destination coding from start coding.


In [ ]:
from importlib import reload
import utils

make_start_end_position_browser = reload(utils).make_start_end_position_browser
if "start_end_browser" in globals():
    for child in start_end_browser.children:
        child.close()
    start_end_browser.close()

start_end_browser = make_start_end_position_browser(movement_bouts, session_info, width=1500)
display(start_end_browser)


# 14. Destination-colored PCA excluding bouts ending near their start

Adjust the minimum absolute net displacement `abs(final_position - initial_position)`.
The initial cutoff is 2 position units: keep bouts with displacement >= 2 and hide
all windows of bouts below that cutoff. A cutoff d corresponds to perpendicular
distance d/sqrt(2) from the identity line in section 13. Set zero to show all bouts
with known endpoints; unknown endpoints are excluded at every cutoff.

This viewer reuses the existing separate PCAs, colored by destination. The slider
only filters displayed points; PCA models, full-session axis ranges, colors and
linked cameras remain unchanged. Panel headings show retained/total observations.


In [ ]:
pca_cache = pd.read_pickle(pca_path)

from importlib import reload
import utils

make_interactive_pca_browser = reload(utils).make_interactive_pca_browser
if "displacement_browser" in globals():
    for child in displacement_browser.children:
        child.close()
    displacement_browser.close()

displacement_browser = make_interactive_pca_browser(
    pca_cache, color_by="destination", filter_displacement=True,
    min_displacement=2.0, width=1400,
)
display(displacement_browser)


# 15. Descriptive linear regression for a continuous label

Predict a chosen numeric label from explicitly selected coordinates. The default
example is starting position, using the selected PCs (currently all) in the last premotor window, fitted
separately for tracking and playback within one session. Each fit uses one
observation per bout. Change the target, dimensions, session or bin below.

The helper fits ordinary least squares with an intercept. R-squared and mean
absolute error are calculated on the same observations used to fit: this is a
description of the recorded data, not held-out prediction or a significance test.
No scaling, PCA fitting or randomization happens here. Numeric categorical codes
such as direction/condition are not continuous targets for this analysis.

Missing/nonfinite values in the chosen target are excluded and counted. Other
labels do not affect inclusion. The prediction table preserves the original
observation index and includes observed/fitted values and residuals.


In [ ]:
pca_cache = pd.read_pickle(pca_path)

from importlib import reload
import evaluation

reload(evaluation)
from evaluation import fit_label_regression, plot_label_regression

REGRESSION_SESSION = 0
REGRESSION_TARGET = "initial_position"
n_pcs = pca_cache["sessions"][REGRESSION_SESSION][0]["model"].n_components_
REGRESSION_COLUMNS = [f"PC{i + 1}" for i in range(3)] # choose a number, or put n_pcs to use all PC's
REGRESSION_BIN = pca_cache["settings"]["N_BINS"] - 1

label_regressions = {}
for condition in (0, 1):
    results = pca_cache["sessions"][REGRESSION_SESSION][condition]["results"]
    mask = results["bin_number"] == REGRESSION_BIN
    label_regressions[condition] = fit_label_regression(
        results.loc[mask, REGRESSION_COLUMNS], results.loc[mask],
        target=REGRESSION_TARGET,
    )

regression_summary = pd.DataFrame([
    {
        "condition": {0: "Tracking", 1: "Playback"}[condition],
        "target": result["target"],
        "n_observations": len(result["predictions"]),
        "n_features": result["n_features"],
        "n_excluded": result["n_excluded"],
        "r2_descriptive": result["r2"],
        "r2_adjusted": result["r2_adjusted"],
        "df_residual": result["df_residual"],
        "mae": result["mae"],
    }
    for condition, result in label_regressions.items()
])
display(regression_summary)
display(label_regressions[0]["predictions"].head())


In [ ]:
import matplotlib.pyplot as plt

fig_regression, axes_regression = plt.subplots(1, 2, figsize=(11, 5))
for ax, (condition, name, color) in zip(
    axes_regression, ((0, "Tracking", "red"), (1, "Playback", "black"))
):
    plot_label_regression(label_regressions[condition], ax=ax, color=color)
    ax.set_title(name + "\n" + ax.get_title())

fig_regression.suptitle(
    f"Session idx={REGRESSION_SESSION}, bin {REGRESSION_BIN}: {REGRESSION_TARGET} from "
    + ", ".join(REGRESSION_COLUMNS)
)
fig_regression.tight_layout()
plt.show()


# 16. The same regression for destination position

Use the same `REGRESSION_SESSION`, `REGRESSION_COLUMNS` and `REGRESSION_BIN`
from section 15, changing only the target to `final_position`. Tracking/playback
are fitted separately using the existing generic helper. Missing endpoints are
excluded and counted. This uses the section 15 subset, without the displacement
filter from section 14. Scores remain descriptive/in-sample, and do not isolate
relationships with destination from those with starting position.


In [ ]:
DESTINATION_TARGET = "final_position"
destination_regressions = {}
for condition in (0, 1):
    results = pca_cache["sessions"][REGRESSION_SESSION][condition]["results"]
    mask = results["bin_number"] == REGRESSION_BIN
    destination_regressions[condition] = fit_label_regression(
        results.loc[mask, REGRESSION_COLUMNS], results.loc[mask],
        target=DESTINATION_TARGET,
    )

destination_regression_summary = pd.DataFrame([
    {
        "condition": {0: "Tracking", 1: "Playback"}[condition],
        "target": result["target"],
        "n_observations": len(result["predictions"]),
        "n_features": result["n_features"],
        "n_excluded": result["n_excluded"],
        "r2_descriptive": result["r2"],
        "r2_adjusted": result["r2_adjusted"],
        "df_residual": result["df_residual"],
        "mae": result["mae"],
    }
    for condition, result in destination_regressions.items()
])
display(destination_regression_summary)


In [ ]:
fig_destination_regression, axes_destination_regression = plt.subplots(1, 2, figsize=(11, 5))
for ax, (condition, name, color) in zip(
    axes_destination_regression, ((0, "Tracking", "red"), (1, "Playback", "black"))
):
    plot_label_regression(destination_regressions[condition], ax=ax, color=color)
    ax.set_title(name + "\n" + ax.get_title())

fig_destination_regression.suptitle(
    f"Destination position | session idx={REGRESSION_SESSION}, bin {REGRESSION_BIN} | "
    f"{len(REGRESSION_COLUMNS)} PCs"
)
fig_destination_regression.tight_layout()
plt.show()


# 17. Destination regression for absolute end-minus-start displacements

Keep bouts satisfying `abs(final_position - initial_position) > n`, including
both directions. Use the same session, PC columns and time window as section 15.
Unknown endpoints are excluded. The original unfiltered analyses remain separate.

Inspect sample counts relative to the number of predictors. Scores describe the
same observations used to fit; no PCA refitting, validation or randomization
is performed. This measures net displacement, not total distance traveled.


In [ ]:
MIN_DISPLACEMENT = 5
large_displacement_regressions = {}

for condition in (0, 1):
    results = pca_cache["sessions"][REGRESSION_SESSION][condition]["results"]
    displacement = (results["final_position"] - results["initial_position"]).abs()
    mask = (results["bin_number"] == REGRESSION_BIN) & (displacement > MIN_DISPLACEMENT)
    selected = results.loc[mask]
    if len(selected) < 2 or selected["final_position"].nunique() < 2:
        print(f"Condition {condition}: insufficient observations or destination variation after filtering.")
        continue
    large_displacement_regressions[condition] = fit_label_regression(
        selected[REGRESSION_COLUMNS], selected, target="final_position",
    )

large_displacement_summary = pd.DataFrame([
    {
        "condition": {0: "Tracking", 1: "Playback"}[condition],
        "n_observations": len(result["predictions"]),
        "n_features": len(REGRESSION_COLUMNS),
        "r2_descriptive": result["r2"],
        "r2_adjusted": result["r2_adjusted"],
        "df_residual": result["df_residual"],
        "mae": result["mae"],
    }
    for condition, result in large_displacement_regressions.items()
])
display(large_displacement_summary)


In [ ]:
fig_large_displacement, axes_large_displacement = plt.subplots(1, 2, figsize=(11, 5))
for ax, (condition, name, color) in zip(
    axes_large_displacement, ((0, "Tracking", "red"), (1, "Playback", "black"))
):
    if condition not in large_displacement_regressions:
        ax.set_title(name)
        ax.text(0.5, 0.5, "Insufficient data after filtering", ha="center", transform=ax.transAxes)
        continue
    plot_label_regression(large_displacement_regressions[condition], ax=ax, color=color)
    ax.set_title(name + "\n" + ax.get_title() + f"\n{len(REGRESSION_COLUMNS)} PCs + intercept")

fig_large_displacement.suptitle(
    f"Destination regression: |end - start| > {MIN_DISPLACEMENT:g} | "
    f"session idx={REGRESSION_SESSION}, bin {REGRESSION_BIN}"
)
fig_large_displacement.tight_layout()
plt.show()


# 18. Run and save one regression per PCA

`run_label_regressions()` pools all premotor windows within each session and
condition. Each PCA gets one regression and one score. Default: destination from
PC1-PC3, retaining absolute end-start displacement >5. Change target, columns,
or cutoff below; use `None` for no displacement filter.

The bundle contains a summary table, regression models with indexed predictions,
and settings. Summary/fit keys are `(idx, condition)`. Both `n_observations`
(bout/window rows) and `n_bouts` (unique bouts) are reported. Predictions keep all
original `(bout_id, bin_start_s)` IDs, without averaging or discarding windows.

Residual degrees of freedom and adjusted R-squared use the regression design's
row count; they do not account for dependence between windows of the same bout.
They are descriptive diagnostics, not a claim of that many independent bouts.
Insufficient subsets are recorded with status/reason. No PCA is refitted.

Files in `data/derived/premotor_activity/evaluation/` have a `_pooled.pkl` suffix.
Reload with `pd.read_pickle(batch_regression_path)` to plot without fitting again.
Each dot in the overview represents one session/condition PCA; horizontal colored
lines show the unweighted session-entry mean. No time-window curve is computed.


In [ ]:
from importlib import reload
from datetime import datetime, timezone
import evaluation

run_label_regressions = reload(evaluation).run_label_regressions
BATCH_TARGET = "final_position"
BATCH_COLUMNS = ["PC1", "PC2", "PC3"]
BATCH_MIN_DISPLACEMENT = 5.0  # None = unfiltered; otherwise keep abs(end-start) > cutoff.

regression_source_cache = pd.read_pickle(pca_path)
batch_regressions = run_label_regressions(
    regression_source_cache, target=BATCH_TARGET, columns=BATCH_COLUMNS,
    min_displacement=BATCH_MIN_DISPLACEMENT,
)
batch_regressions["settings"]["saved_at_utc"] = datetime.now(timezone.utc).isoformat()

evaluation_dir.mkdir(parents=True, exist_ok=True)
filter_tag = "all_bouts" if BATCH_MIN_DISPLACEMENT is None else f"absdelta_gt{BATCH_MIN_DISPLACEMENT:g}"
batch_regression_path = evaluation_dir / (
    f"regression_{BATCH_TARGET}_{'-'.join(BATCH_COLUMNS)}_{filter_tag}_{N_BINS}bins_pooled.pkl"
)
pd.to_pickle(batch_regressions, batch_regression_path)

print(f"Saved {len(batch_regressions['fits'])} regressions to {batch_regression_path}")
print(batch_regressions["summary"]["status"].value_counts())
print("Fits with no residual degrees of freedom:", int((batch_regressions["summary"]["df_residual"] <= 0).sum()))
display(batch_regressions["summary"].head(10))
if (batch_regressions["summary"]["status"] == "skipped").any():
    display(batch_regressions["summary"].loc[batch_regressions["summary"]["status"] == "skipped"])


In [ ]:
import matplotlib.pyplot as plt

plot_regression_summary = evaluation.plot_regression_summary
fig_batch_regression, axes_batch_regression = plot_regression_summary(
    batch_regressions["summary"], score="r2_adjusted",
)
saved_settings = batch_regressions["settings"]
saved_cutoff = saved_settings["min_displacement"]
saved_filter_label = "all bouts" if saved_cutoff is None else f"abs(end-start) > {saved_cutoff:g}"
fig_batch_regression.suptitle(
    f"{saved_settings['target']} from {', '.join(saved_settings['columns'])} | {saved_filter_label} | all windows pooled | {ACTIVITY_MODE}"
)
fig_batch_regression.tight_layout()
plt.show()


# 19. Paired tracking/playback score comparison

Match the two pooled regression scores by session `idx`. The left panel connects
each session's tracking (red) and playback (black) scores. The right panel shows
tracking minus playback for each session; positive values mean higher tracking
scores. Only complete pairs with successful fits and finite scores are included.

Use `PAIR_SCORE = "r2_adjusted"` to match section 18, or `"r2"` for ordinary
R-squared. This reads saved regression scores without fitting models. Differences
and their mean are descriptive; no significance test or confidence interval is added.


In [ ]:
from importlib import reload
import evaluation
import matplotlib.pyplot as plt

plot_paired_regressions = reload(evaluation).plot_paired_regressions
PAIR_SCORE = "r2_adjusted"  # "r2" = ordinary R-squared.
fig_paired_regression, axes_paired_regression, paired_regression_scores = plot_paired_regressions(
    batch_regressions["summary"], score=PAIR_SCORE,
)
fig_paired_regression.suptitle(
    f"{batch_regressions['settings']['target']} | {PAIR_SCORE} | all windows pooled | {ACTIVITY_MODE}"
)
fig_paired_regression.tight_layout()
plt.show()
print(f"Complete pairs: {len(paired_regression_scores)} / {batch_regressions['summary'].index.get_level_values('idx').nunique()} session entries")
display(paired_regression_scores)


# 20. Pooled regression for starting position, using all bouts

Use the same source PCAs and feature columns as section 18, with target
`initial_position` and no displacement filter. All bouts with a finite starting
position are included, regardless of destination or movement size. All premotor
windows are pooled into one fit per session/condition.

Destination results remain in `batch_regressions`; starting-position results
are stored separately. Run after section 18, or reload its bundle/settings and
source PCA cache. The plots default to adjusted R-squared; change
`STARTING_POSITION_SCORE` to `"r2"` for ordinary R-squared.


In [ ]:
from importlib import reload
from datetime import datetime, timezone
import evaluation

reload(evaluation)
regression_source_cache = pd.read_pickle(pca_path)
comparison_settings = batch_regressions["settings"]
starting_position_regressions = evaluation.run_label_regressions(
    regression_source_cache, target="initial_position",
    columns=comparison_settings["columns"],
    min_displacement=None,  # Starting-position analysis includes all bouts.
)
starting_position_regressions["settings"]["saved_at_utc"] = datetime.now(timezone.utc).isoformat()

start_settings = starting_position_regressions["settings"]
start_cutoff = start_settings["min_displacement"]
start_filter_tag = "all_bouts" if start_cutoff is None else f"absdelta_gt{start_cutoff:g}"
starting_position_path = evaluation_dir / (
    f"regression_initial_position_{'-'.join(start_settings['columns'])}_{start_filter_tag}_{N_BINS}bins_pooled.pkl"
)
pd.to_pickle(starting_position_regressions, starting_position_path)
print(f"Saved {len(starting_position_regressions['fits'])} starting-position regressions to {starting_position_path}")
print(starting_position_regressions["summary"]["status"].value_counts())
display(starting_position_regressions["summary"].head())


In [ ]:
import matplotlib.pyplot as plt

STARTING_POSITION_SCORE = "r2_adjusted"  # "r2" = ordinary R-squared.
fig_start_overview, axes_start_overview = evaluation.plot_regression_summary(
    starting_position_regressions["summary"], score=STARTING_POSITION_SCORE,
)
fig_start_overview.suptitle(
    f"Starting position from {', '.join(start_settings['columns'])} | all bouts | all windows pooled | {ACTIVITY_MODE}"
)
fig_start_overview.tight_layout()
plt.show()


In [ ]:
fig_start_paired, axes_start_paired, starting_position_pairs = evaluation.plot_paired_regressions(
    starting_position_regressions["summary"], score=STARTING_POSITION_SCORE,
)
fig_start_paired.suptitle(f"Starting position | {STARTING_POSITION_SCORE} | all windows pooled | {ACTIVITY_MODE}")
fig_start_paired.tight_layout()
plt.show()
print(f"Complete pairs: {len(starting_position_pairs)}")
print(f"Mean tracking - playback: {starting_position_pairs['difference'].mean():.4f}")
display(starting_position_pairs)


# 21. Starting-position scores versus side preference and position spread

Use the exact observations retained in the starting-position regressions. Count
each bout once when describing its starting-position distribution, rather than
counting its five premotor windows as five bouts. This uses all bouts from
section 20, with no displacement filter or additional selection.

At midpoint 14, left is position <14 and right is >=14. Signed side bias is
`(n_right - n_left) / n_bouts`; its magnitude is 0 for balanced sides and 1 for
one-sided data. Position spread is the sample standard deviation of unique-bout
starting positions. Conventional statistical skewness is recorded in the table
for reference, but is distinct from preference relative to the physical midpoint.

The four-panel graph compares scores with absolute side bias (top) and position
spread (bottom), separately for tracking/playback. The paired graph compares
tracking-minus-playback differences in score and absolute side bias. Positive x
means tracking is more one-sided; positive y means its regression score is higher.
These are descriptive checks; no regression, PCA, or significance test is run.


In [ ]:
from importlib import reload
import evaluation

reload(evaluation)
POSITION_MIDPOINT = 14.0
POSITION_BIAS_SCORE = "r2_adjusted"  # "r2" = ordinary R-squared.
position_bias_summary = evaluation.summarize_starting_position_bias(
    starting_position_regressions, midpoint=POSITION_MIDPOINT,
)
position_bias_summary.attrs["midpoint"] = POSITION_MIDPOINT
position_bias_summary.attrs["source_regression_file"] = str(starting_position_path)
position_bias_path = starting_position_path.with_name(starting_position_path.stem + "_position_bias.pkl")
position_bias_summary.to_pickle(position_bias_path)
print(f"Saved bout-level distribution measures and existing regression scores to {position_bias_path}")
display(position_bias_summary[[
    "n_position_bouts", "n_left", "n_right", "side_bias", "abs_side_bias",
    "position_mean", "position_std", "position_skewness", POSITION_BIAS_SCORE,
]])


In [ ]:
import matplotlib.pyplot as plt

fig_position_bias, axes_position_bias = evaluation.plot_position_bias_scores(
    position_bias_summary, score=POSITION_BIAS_SCORE,
)
fig_position_bias.suptitle(f"Starting-position regression | each position counted once per bout | {ACTIVITY_MODE}")
fig_position_bias.tight_layout()
plt.show()


In [ ]:
fig_paired_bias, ax_paired_bias, paired_bias_scores = evaluation.plot_paired_bias_differences(
    position_bias_summary, score=POSITION_BIAS_SCORE,
)
fig_paired_bias.suptitle(f"Starting-position regression | {ACTIVITY_MODE}")
fig_paired_bias.tight_layout()
plt.show()
display(paired_bias_scores)


# 22. Left/right logistic regression across sessions

Fit one classifier per session and condition from the saved PCA coordinates.
Use `initial_direction`: -1 = leftward movement, +1 = rightward movement;
exclude 0/missing/unknown directions. This describes movement direction, rather
than which side the bout starts on. Include all bouts with known direction,
with all premotor windows pooled and no displacement filter.

Default features are PC1-PC3. Each fit uses L2 logistic regression (`C=1`),
an intercept, and equal total class weights for left/right. Selected coordinates
are standardized within the classifier; the scaler and classifier are saved
together. The source PCAs are reused without refitting.

Balanced accuracy is the mean of left and right recall. The 0.5 reference is
the score for always predicting one direction, not a significance threshold.
These are descriptive scores on the fitted observations, consistent with our
position regressions. Windows from one bout remain related observations.

The first graph shows one score per session, separately for tracking/playback.
The second matches conditions by session idx and shows paired scores plus
tracking-minus-playback differences. Positive differences favor tracking.


In [ ]:
from importlib import reload
from datetime import datetime, timezone
import evaluation

reload(evaluation)
n_pcs = pca_cache["sessions"][REGRESSION_SESSION][0]["model"].n_components_
DIRECTION_COLUMNS = [f"PC{i + 1}" for i in range(3)] # choose a number, or put n_pcs to use all PC's
DIRECTION_C = 1.0  # Inverse L2 penalty strength; larger = weaker penalty.

direction_source_path = pca_path
direction_source_cache = pd.read_pickle(direction_source_path)
direction_logistics = evaluation.run_direction_logistics(
    direction_source_cache, columns=DIRECTION_COLUMNS, C=DIRECTION_C,
)
direction_logistics["settings"]["source_pca_file"] = str(direction_source_path)
direction_logistics["settings"]["saved_at_utc"] = datetime.now(timezone.utc).isoformat()

evaluation_dir.mkdir(parents=True, exist_ok=True)
direction_logistic_path = evaluation_dir / (
    f"logistic_initial_direction_{'-'.join(DIRECTION_COLUMNS)}_C{DIRECTION_C:g}_{N_BINS}bins_pooled.pkl"
)
pd.to_pickle(direction_logistics, direction_logistic_path)
print(f"Saved {len(direction_logistics['fits'])} direction classifiers to {direction_logistic_path}")
print(direction_logistics["summary"]["status"].value_counts())
display(direction_logistics["summary"].head(10))
if (direction_logistics["summary"]["status"] == "skipped").any():
    display(direction_logistics["summary"].loc[direction_logistics["summary"]["status"] == "skipped"])


In [ ]:
import matplotlib.pyplot as plt

fig_direction_overview, axes_direction_overview = evaluation.plot_regression_summary(
    direction_logistics["summary"], score="balanced_accuracy",
)
fig_direction_overview.suptitle(
    f"Left/right from {', '.join(DIRECTION_COLUMNS)} | descriptive balanced accuracy | all windows pooled | {ACTIVITY_MODE}"
)
fig_direction_overview.tight_layout()
plt.show()


In [ ]:
fig_direction_paired, axes_direction_paired, direction_pairs = evaluation.plot_paired_regressions(
    direction_logistics["summary"], score="balanced_accuracy",
)
fig_direction_paired.suptitle(f"Left/right | descriptive balanced accuracy | all windows pooled | {ACTIVITY_MODE}")
fig_direction_paired.tight_layout()
plt.show()
print(f"Complete pairs: {len(direction_pairs)}")
print(f"Mean tracking - playback: {direction_pairs['difference'].mean():.4f}")
display(direction_pairs)
